In [11]:
import pandas as pd
import os
import re
from pathlib import Path

os.chdir ('C:\\Users\\hamod4691\\Code\\arctic-benthic-biodiversity')

# Fix mojibake

In [71]:
def _fix_mojibake_text(x):
    if not isinstance(x, str):
        return x

    s = x
    if not any(ch in s for ch in ("Ã", "Â", "¤", "¦", "¥", "œ", "�")):
        return s

    candidates = [s]
    for enc in ("latin1", "cp1252"):
        try:
            candidates.append(s.encode(enc).decode("utf-8"))
        except Exception:
            pass

    def score(t):
        bad = sum(t.count(ch) for ch in ("Ã", "Â", "�"))
        return (bad, -sum(ord(ch) > 127 for ch in t))

    best = min(candidates, key=score)
    return best

p = 'AU/bio_data.csv'
df = pd.read_csv(p)
df_edited = df.copy()

for col in df_edited.select_dtypes(include=["object"]).columns:
    df_edited[col] = df_edited[col].map(_fix_mojibake_text)

out_p = os.path.join(os.path.dirname(p), "bio_data_edited.csv")
df_edited.to_csv(out_p, index=False, encoding="utf-8-sig")

print(f"Saved: {out_p}")

Saved: AU\bio_data_edited.csv


C:\Users\hamod4691\AppData\Local\Temp\ipykernel_17784\3691843218.py:27: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  for col in df_edited.select_dtypes(include=["object"]).columns:


# Move identificationQualifier

In [ ]:
p = 'AU/bio_data_edited.csv'
df = pd.read_csv(p)

qualifier_pattern = re.compile(
    r'\([^)]*\)|'                       # parentheses (keep them)
    r'(?i:\bPamiut|NKF\b)|'             # specific place-name qualifier
    r'(?i:\bsp\.?\d+\b)|'             # sp2 or sp.2
    r'(?i:(?<=\s)\.?juv\.?)|(?i:(?<=\s)juv\.?)(?=\W|$)|'  # juv preceded by space, dot optional
    r'(?i:gen\. et spp\.?)|'            # gen. et spp.
    r'(?i:\b(?:spp|sp|indet|lign|stage|conf|var)\.?)(?=\W|$)|'  # spp/sp/indet/lign/conf/var
    r'\bK\d(?:-\d){2}\b|'             # K-codes like K3-2-3
    r'\b[A-Z]{1,}[0-9]+(?:-[0-9]+)?\b|' # codes like NKF22-4, DW31-1, T-1 (letters+digits)
    r'\b[A-Z](?:-\d+|\d+)?\b|'        # lone capital letters maybe with -number (e.g. T-1)
    r'\b\d+\b|'                       # standalone numbers
    r'\?+'                              # question marks
)


def _extract_qualifiers(text):
    matches = list(qualifier_pattern.finditer(text))
    qualifiers = [match.group(0).strip() for match in matches]

    parts = []
    last_end = 0
    for match in matches:
        parts.append(text[last_end:match.start()])
        last_end = match.end()
    parts.append(text[last_end:])

    cleaned = re.sub(r'\s+', ' ', ''.join(parts)).strip(' ,;.-')
    if cleaned == '':
        cleaned = pd.NA

    return cleaned, qualifiers


def split_qualifier(s):
    if pd.isna(s):
        return pd.Series([pd.NA, pd.NA])

    orig = str(s)
    if '/' in orig:
        left, right = orig.split('/', 1)
        cleaned, qualifiers = _extract_qualifiers(left)
        right_cleaned, right_qualifiers = _extract_qualifiers(right)
        qualifiers.extend(right_qualifiers)
        if pd.notna(right_cleaned):
            qualifiers.append(str(right_cleaned).strip())
    else:
        cleaned, qualifiers = _extract_qualifiers(orig)

    qual = '; '.join(q for q in qualifiers if q and q.strip()) if qualifiers else pd.NA
    return pd.Series([cleaned, qual])


df[['scientificName', 'identificationQualifier']] = df['Taxon edited'].apply(split_qualifier)
# write CSV (may raise if file is locked)
df.to_csv(p, index=False)
# also write Excel
try:
    df.to_excel(os.path.splitext(p)[0] + '.xlsx', index=False)
except Exception as e:
    print('Excel write failed:', e)

# Sort out names

In [ ]:
p = 'AU/bio_data_edited.csv'
df = pd.read_csv(p)

def _save_outputs():
    df.to_csv(p, index=False, encoding='utf-8-sig')
    df.to_excel(os.path.splitext(p)[0] + '.xlsx', index=False)

rename_pairs = [
    ('Ampharete finmarchia', 'Ampharete finmarchica'),
    ('Ampelisca eschrichti', 'Ampelisca eschrichtii'),
    ('Ampharete goesi', 'Ampharete goësi'),
    ('Amphitite', 'Amphitrite'),
    ('Bivalvia ånderør', 'Bivalvia'),
    ('Byblis gaimardi', 'Byblis gaimardii'),
    ('Bylgides groenlandica', 'Bylgides groenlandicus'),
    ('Cingula moerchi', 'Alvania moerchii'),
    ('Cingula moercki', 'Alvania moerchii'),
    ('Cossura longicirrata', 'Cossura longocirrata'),
    ('Diastylis scorpipides', 'Diastylis scorpioides'),
    ('Microphthalmus sczelkowi', 'Microphthalmus sczelkowii'),
    ('Monoculodes kroyeri', 'Rostroculodes kroyeri'),
    ('Monoculodes krøyeri', 'Rostroculodes kroyeri'),
    ('Mya ånderør', 'Mya sp.'),
    ('Nainereis quadricuspida', 'Naineris quadricuspida'),
    ('Nebalia biceps', 'Nebalia bipes'),
    ('Nephtys longosetosus', 'Nephtys longosetosa'),
    ('Nicomache trispina', 'Nicomache trispinata'),
    ('Nudibranch', 'Nudibranchia'),
    ('Octocoral', 'Octocorallia'),
    ('Oediceros sagnisatus', 'Oediceros saginatus'),
    ('Oediceridae', 'Oedicerotidae'),
    ('Oenopota turricola', 'Propebela turricula'),
    ('Oenopota turricula', 'Propebela turricula'),
    ('Ophiocanta bidentata', 'Ophiacantha bidentata'),
    ('Ophiura sarsi', 'Ophiura sarsii'),
    ('Opisa eschrichti', 'Opisa eschrichtii'),
    ('Pholoë', 'Pholoe'),
    ('Pholoë', 'Pholoe'),
    ('Phoxocephalus holboelli', 'Phoxocephalus holbolli'),
    ('Phoxocephalus holbølli', 'Phoxocephalus holbolli'),
    ('Sphaerodoropsis minuta', 'Sphaerodoridium minutum'),
    ('Tachyrhyncus reticulatus', 'Tachyrhynchus reticulatus'),
    ('Thyasira gouldi', 'Thyasira gouldii'),
    ('Travisia forbesi', 'Travisia forbesii'),
    ('Urothoe elegans', 'Urthoe elegans'),
    ('Yoldia hyperboria', 'Yoldia hyperborea'),
]

for old_name, new_name in rename_pairs:
    mask = df['scientificName'].astype(str).str.strip().eq(old_name)
    count = mask.sum()
    df.loc[mask, 'scientificName'] = new_name
    _save_outputs()
    print(f"Updated {count} rows and saved:\n- {p}\n- {os.path.splitext(p)[0] + '.xlsx'}")

Removed 4 rows and saved:
- AU/bio_data_edited.csv
- AU/bio_data_edited.xlsx
Updated 10 rows and saved:
- AU/bio_data_edited.csv
- AU/bio_data_edited.xlsx
Updated 32 rows and saved:
- AU/bio_data_edited.csv
- AU/bio_data_edited.xlsx
Updated 15 rows and saved:
- AU/bio_data_edited.csv
- AU/bio_data_edited.xlsx
Updated 1 rows and saved:
- AU/bio_data_edited.csv
- AU/bio_data_edited.xlsx
Updated 4 rows and saved:
- AU/bio_data_edited.csv
- AU/bio_data_edited.xlsx
Updated 2 rows and saved:
- AU/bio_data_edited.csv
- AU/bio_data_edited.xlsx
Updated 2 rows and saved:
- AU/bio_data_edited.csv
- AU/bio_data_edited.xlsx
Updated 1 rows and saved:
- AU/bio_data_edited.csv
- AU/bio_data_edited.xlsx
Updated 16 rows and saved:
- AU/bio_data_edited.csv
- AU/bio_data_edited.xlsx
Updated 2 rows and saved:
- AU/bio_data_edited.csv
- AU/bio_data_edited.xlsx
Updated 1 rows and saved:
- AU/bio_data_edited.csv
- AU/bio_data_edited.xlsx
Updated 1 rows and saved:
- AU/bio_data_edited.csv
- AU/bio_data_edited.

# Get aphiaIDs

In [74]:
unique_names = (
    df["scientificName"]
    .dropna()
    .astype(str)
    .str.strip()
    .loc[lambda s: s.ne("")]
    .drop_duplicates()
    .sort_values()
    .reset_index(drop=True)
)

unique_names_df = pd.DataFrame({"scientificName": unique_names})
unique_names_df.to_csv("AU/unique_names.csv", index=False, encoding="utf-8-sig")

print(f"Saved {len(unique_names_df)} unique names to unique_names.csv")
unique_names_df.head()

Saved 1144 unique names to unique_names.csv


,scientificName
0,Abyssoninoe
1,Abyssoninoe abyssorum
2,Acanthonotozoma magnum
3,Acanthonotozoma rusanovae
4,Acanthonotozoma serratum


In [2]:
bio_p = "AU/bio_data_edited.xlsx"
match_p = "AU/unique_names_matched.xlsx"

bio_df = pd.read_excel(bio_p)
match_df = pd.read_excel(match_p)

name_to_aphia = (
    match_df[["scientificName", "AphiaID_accepted"]]
    .dropna(subset=["scientificName"])
    .assign(scientificName=lambda d: d["scientificName"].astype(str).str.strip())
    .drop_duplicates(subset=["scientificName"], keep="last")
    .set_index("scientificName")["AphiaID_accepted"]
)

bio_df["AphiaID_accepted"] = (
    bio_df["scientificName"]
    .astype(str)
    .str.strip()
    .map(name_to_aphia)
)

bio_df.to_excel(bio_p, index=False)

print(f"Saved: {bio_p}")

Saved: AU/bio_data_edited.xlsx


Go through and fill in the blanks manually

In [8]:
# Read the manual file
bio_manual_p = "AU/bio_data_edited_manual.xlsx"
bio_manual_df = pd.read_excel(bio_manual_p)

# Remove records where AphiaID_accepted is NaN
bio_filtered_df = bio_manual_df.dropna(subset=['AphiaID_accepted'])

# Save the filtered dataframe
bio_filtered_p = "AU/bio_data_filtered.xlsx"
bio_filtered_df.to_excel(bio_filtered_p, index=False)

print(f"Removed {len(bio_manual_df) - len(bio_filtered_df)} records without AphiaID_accepted")
print(f"Saved {len(bio_filtered_df)} records to {bio_filtered_p}")

Removed 7 records without AphiaID_accepted
Saved 24745 records to AU/bio_data_filtered.xlsx


# Key in station data

In [12]:
bio_filtered_p = "AU/bio_data_filtered.xlsx"

# Read the filtered bio data
bio_filtered_df = pd.read_excel(bio_filtered_p)

# Read the stations data
stations_df = pd.read_csv('AU/stations_1.csv', sep=';')
stations_df.columns = stations_df.columns.str.strip()

# Normalize station_id in bio_filtered_df to match stations format
# Convert to string and strip whitespace for comparison
bio_filtered_df['station_id'] = bio_filtered_df['station_id'].astype(str).str.strip()
stations_df['Station ID'] = stations_df['Station ID'].astype(str).str.strip()

# Create a case-insensitive composite key for merging
bio_filtered_df['merge_key'] = (
    bio_filtered_df['Cruise ID'].astype(str).str.strip().str.casefold()
    + '|'
    + bio_filtered_df['station_id'].astype(str).str.strip().str.casefold()
)
stations_df['merge_key'] = (
    stations_df['Cruise ID'].astype(str).str.strip().str.casefold()
    + '|'
    + stations_df['Station ID'].astype(str).str.strip().str.casefold()
)

# Merge on the composite key
bio_full_df = bio_filtered_df.merge(
    stations_df[['merge_key', 'Month', 'Day', 'Date', 'Depth', 'decimalLatitudeStart', 'decimalLongitudeStart', 'sampling equip']],
    on='merge_key',
    how='left'
)

# Drop the temporary merge key column
bio_full_df = bio_full_df.drop('merge_key', axis=1)

# Save to Excel
bio_full_df.to_excel('AU/bio_data_full.xlsx', index=False)

# Check coverage for the coordinate columns in the saved table
coordinate_coverage = bio_full_df[['decimalLatitudeStart', 'decimalLongitudeStart']].notna().mean().mul(100).round(2)
print(f"Saved AU/bio_data_full.xlsx with {len(bio_full_df)} records")
print('decimalLatitudeStart coverage:', f"{coordinate_coverage['decimalLatitudeStart']}%")
print('decimalLongitudeStart coverage:', f"{coordinate_coverage['decimalLongitudeStart']}%")
bio_full_df.head()

Saved AU/bio_data_full.xlsx with 24968 records
decimalLatitudeStart coverage: 96.35%
decimalLongitudeStart coverage: 96.35%


,Number id,Cruise ID,station_id,Station_old,Delprøve,Redskab,Number of subsamples,Group,Year,Taxon edited,...,scientificName,identificationQualifier,AphiaID_accepted,Month,Day,Date,Depth,decimalLatitudeStart,decimalLongitudeStart,sampling equip
0,14580,MAARMORILIK,1,NaN,1,VV,NaN,CRUSTACEA,2005,megalopedia,...,NaN,NaN,1066,NaN,NaN,NaN,55.0,71.106667,-51.233333,Haps
1,71,KANUMAS,1.1,1-Jan,3,VV,NaN,ANIMALIA,2008,Animalia indet.,...,Animalia,indet.,2,8.0,13.0,8/13/2008,21.0,76.920617,-70.907383,Van Veen
2,994,KANUMAS,1.1,1-Oct,1,VV,NaN,ANIMALIA,2008,Animalia indet.,...,Animalia,indet.,2,8.0,13.0,8/13/2008,21.0,76.920617,-70.907383,Van Veen
3,1000,KANUMAS,1.1,1-Oct,5,VV,NaN,GASTROPODA,2008,Gastropoda indet.,...,Gastropoda,indet.,101,8.0,13.0,8/13/2008,21.0,76.920617,-70.907383,Van Veen
4,113,KANUMAS,1.2,2-Jan,1,VV,NaN,ANIMALIA,2008,Animalia indet.,...,Animalia,indet.,2,8.0,14.0,8/14/2008,65.0,76.915600,-71.061500,Van Veen


In [14]:
p = "AU/bio_data_full.xlsx"

# Use the merged dataframe if available; otherwise read from disk
try:
    df_work = bio_full_df.copy()
except NameError:
    df_work = pd.read_excel(p)

# Count records before filtering
count_before = len(df_work)

# Remove records without both latitude and longitude
df_work = df_work.dropna(subset=['decimalLatitudeStart', 'decimalLongitudeStart'])

# Count records after filtering
count_after = len(df_work)

# Save with the same name
df_work.to_excel(p, index=False)

# Print summary
print(f"Records before: {count_before}")
print(f"Records after: {count_after}")
print(f"Removed: {count_before - count_after}")
print(f"Saved: {p}")

Records before: 24968
Records after: 24056
Removed: 912
Saved: AU/bio_data_full.xlsx


# Key in flaggedComment

# Create startDayOfYear column

In [15]:
p = "AU/bio_data_full.xlsx"

# Use the in-memory dataframe if available; otherwise read from disk
try:
    df = bio_full_df.copy()
except NameError:
    df = pd.read_excel(p)

date_vals = pd.to_datetime(
    dict(
        year=pd.to_numeric(df["Year"], errors="coerce"),
        month=pd.to_numeric(df["Month"], errors="coerce"),
        day=pd.to_numeric(df["Day"], errors="coerce"),
    ),
    errors="coerce",
)

df["startDayOfYear"] = date_vals.dt.dayofyear

df.to_excel(p, index=False)
print(f"Saved: {p}")

Saved: AU/bio_data_full.xlsx


# Prepare for merging

In [ ]:
au = pd.read_excel("AU/bio_data_full.xlsx")

dtype_spec = pd.read_excel(Path("..") / "column_data_types.xlsx", sheet_name=0)
dtype_map = dict(zip(dtype_spec['column'], dtype_spec['final']))

column_map = {
    "sourceID": "Number id",
    "datasetKey": "Cruise ID",
    "decimalLatitudeStart": "decimalLatitudeStart",
    "decimalLongitudeStart": "decimalLongitudeStart",
    "depthA_m": "Depth",
    "eventID": "station_id",
    "locationID": "station_id",
    "verbatimDate": "Date",
    "startDayOfYear": "startDayOfYear",
    "year": "Year",
    "verbatimScientificName": "Taxon edited",
    "identificationQualifier": "identificationQualifier",
    "aphiaID": "AphiaID_accepted",
    "individualCount": "Abundance",
    "biomass": "Wet weight per sample",
    "gear": "sampling equip"
}

output_columns = [
    "ID",
    "sourceID",
    "verbatimID",
    "sourceDatabase",
    "datasetKey",
    "datasetTitle",
    "issue",
    "decimalLatitudeStart",
    "decimalLongitudeStart",
    "decimalLatitudeStop",
    "decimalLongitudeStop",
    "footprintWKT",
    "coordinateUncertaintyInMeters",
    "hasGeospatialIssues",
    "depthA_m",
    "depthAType",
    "depthB_m",
    "depthBType",
    "gebcoDepth",
    "eventID",
    "parentEventID",
    "waterBody",
    "largeMarineEcosystem",
    "locationID",
    "verbatimDate",
    "startDayOfYear",
    "endDayOfYear",
    "year",
    "eventRemarks",
    "verbatimScientificName",
    "acceptedScientificName",
    "identificationQualifier",
    "lifeStage",
    "sex",
    "taxonomicStatus",
    "taxonomicIssue",
    "taxonRank",
    "phylum",
    "class",
    "order",
    "family",
    "genus",
    "specificEpithet",
    "validAuthority",
    "aphiaID",
    "individualCount",
    "areaSampled",
    "areaSampledUnit",
    "samplingEffort",
    "percentCoverage",
    "biomass",
    "biomassUnit",
    "dynamicProperties",
    "basisOfRecord",
    "occurrenceRemarks",
    "gear",
    "gearCategory",
    "meshSize_mm",
    "sieveFraction_µm",
    "flagged",
    "flaggedComment", 
    "acceptedTaxonKey"
]


def normalize_dtype(dtype_name):
    dtype_name = str(dtype_name).strip().lower()
    return {
        "int32": "Int32",
        "float32": "Float32",
        "float64": "Float64",
        "boolean": "boolean",
        "string": "string",
    }.get(dtype_name, dtype_name)


def empty_series(index, dtype_name):
    dtype_name = normalize_dtype(dtype_name)
    if dtype_name in {"Int32", "Float32", "Float64", "boolean", "string"}:
        return pd.Series(pd.NA, index=index, dtype=dtype_name)
    return pd.Series(pd.NA, index=index, dtype="string")


def coerce_series(series, dtype_name):
    dtype_name = normalize_dtype(dtype_name)
    if dtype_name == "boolean":
        return series.astype("boolean")
    if dtype_name in {"Int32", "Float32", "Float64"}:
        return pd.to_numeric(series, errors="coerce").astype(dtype_name)
    return series.astype("string")


ogsl_final = pd.DataFrame(index=ogsl.index)
missing_sources = []

for col in output_columns:
    target_dtype = dtype_map.get(col, "string")
    if col in column_map:
        src = column_map[col]
        if src in ogsl.columns:
            ogsl_final[col] = coerce_series(ogsl[src], target_dtype)
        else:
            ogsl_final[col] = empty_series(ogsl.index, target_dtype)
            missing_sources.append(src)
    else:
        ogsl_final[col] = empty_series(ogsl.index, target_dtype)

ogsl_final = ogsl_final[output_columns]
ogsl_final.to_parquet("ogsl_final.parquet", index=False)

print(f"Saved ogsl_final.parquet with {len(ogsl_final):,} rows and {len(ogsl_final.columns)} columns.")
if missing_sources:
    print("Missing source columns in ogsl:", sorted(set(missing_sources)))